In [ ]:
# today we build the unified Research agent
"""Today I stop building isolated tools and fuse them into one agent that can decide, on it's own
Whether to search my local PDFs or the live web.

Tool roster: Combine everything from day 1-6 into a single tools list- my local RAG
retriever (vector search over my PDFs), your Tavily web search tool, and your Validator
node from day 6 wrapping all of them.

Routing Logic: The agent shouldn't be told which tool to use , it must decide. Am going
to write the system prompt so that Llama 3.3 chooses search_local_docs for questions about
ingested pdfs and web_search for anything current or outside my corpus.

Multi-tool Chaining: Handle the case where one query needs both - e.g.
"compare with what my PDF says about apple releases to what's being published this year". the 
agent should call both tools and synthesise

"""

# assembling the full tool rooster
import os
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from dotenv import load_dotenv, find_dotenv
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


load_dotenv(find_dotenv())


llm = ChatGroq(
    model= 'llama-3.3-70B-versatile',
    api_key= os.getenv('GROQ_API_KEY'),
    temperature= 0
)

# setting embedding model
Settings.embed_model = HuggingFaceEmbedding(
    model_name= 'BAAI/bge-m3'
)


#---Tool 2: Live web search---
web_search = TavilySearchResults(max_results = 3)
web_search.name = "web_search"

# --- Tool 3: Reuse Day 6's validated tool as an example domain tool ---
"""@tool
def get_apple_docs() -> str:
    '''Returns a list of available things to search from according to the apple doc.'''
    return "Available years are 2021,2022 and 2023"

"""

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\rodne\AppData\Local\Temp\ipykernel_21780\4090533882.py:46: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  web_search = TavilySearchResults(max_results = 3)


'@tool\ndef get_apple_docs() -> str:\n    \'\'\'Returns a list of available things to search from according to the apple doc.\'\'\'\n    return "Available years are 2021,2022 and 2023"\n\n'

In [ ]:
# setting up vector db connection before i set up the vector engine 
from llama_index.vector_stores.mongodb import MongoDBAtlasVectorSearch
from llama_index.storage.docstore.mongodb import MongoDocumentStore
from pymongo import MongoClient, AsyncMongoClient
from llama_index.core import VectorStoreIndex, StorageContext

# Re-establishing the connection to my mongoDB atlas vector store to retrieve historical data

# mongoDB atlas connection
mongoClient = MongoClient(os.getenv('MONGO_URI'))
asyncMongoClient = AsyncMongoClient(os.getenv('MONGO_URI'))


# reconnecting persistent docstore
docstore = MongoDocumentStore.from_uri(
    uri = os.getenv('MONGO_URI'),
    db_name = 'month1_database',
    namespace = 'month_1_collection'
)

# 1. Connect to your MongoDB
vectorStore = MongoDBAtlasVectorSearch(
    mongodb_client=mongoClient,
    async_mongodb_client= asyncMongoClient, 
    db_name= 'month1_database',
    collection_name= 'month_1_rag_collection_v3',
    vector_index_name= 'final_index_v3',
    embedding_key= 'embedding'
)

storageContext = StorageContext.from_defaults(
    vector_store= vectorStore,
    docstore= docstore
)

index = VectorStoreIndex.from_vector_store(
    vector_store= vectorStore,
    storage_context = storageContext
)



print('🤖🛩️ Vector Store connection established ⚡')

🤖🛩️ Vector Store connection established ⚡


In [ ]:
#---Tool 1: Local RAG retriever(reusing my month 1 vector store) ---
@tool 
def search_local_docs(query:str) -> str:
    """Searches my local Knowledge base containing historical Apple 10-K financial documents
    (covering fiscal years up to 2024). Use this to retrieve historical sales, net revenue,
    and internal corporate performance figures.
    
    CRITICAL: Do not pass comparative or converstional questions here.
    Convert queries into strict financial line items, such as:
    - 'Apple consolidated statements of operations net sales' 
    - 'Apple total net sales 2023 -2024' 
    - 'Summary of operations data'
    """

    retriever = index.as_retriever(similarity_top_k = 4)
    results = retriever.retrieve(query)
    return "\n\n".join([doc.node.get_content() for doc in results])


tools = [search_local_docs, web_search]
toolMap = {t.name: t for t in tools} 

llmWithTools = llm.bind_tools(tools= tools)


In [5]:
# reusing the validator from day 6
def validate_and_execute(tool_call: dict) -> str:
    toolName = tool_call['name']
    if toolName not in toolMap:
        error_msg = (
            f"SYSTEM ERROR: You attempted to call a tool named '{toolName}', "
            f"but it does not exist. You can ONLY use the following tools: {list(toolMap.keys())}. "
            f"Please think step-by-step and try again."
        )
        print(f"🤥 VALIDATOR CAUGHT ERROR: {error_msg}")
        return error_msg
    try:
        print(f"😁VALIDATOR PASSED: Executing '{toolName}' ...")
        return toolMap[toolName].invoke(tool_call['args'])
    except Exception as e:
        error_msg = f"SYSTEM ERROR  executing '{toolName}': {str(e)}. Please correct your arguments."
        print(f"🤥 VALIDATOR CAUGHT ERROR: {error_msg}")
        return error_msg

In [ ]:
# finally the research agent loop
def run_research_agent(query: str, max_retries: int = 3):
    # explicitly telling the model how to output tool payloads
    # FORCING THE MODEL TO PLAN AND NATIVELY EXECUTE TOOLS

    messages = [
        SystemMessage(content=("You are a precise dual-retrieval research assistant.\n"
            "CRITICAL GUIDELINES:\n"
            "1. If a query asks to compare historical data with recent, current, or 'last year' figures,"
            "you MUST use both tools: use 'search_local_docs' for historical data in your files,"
            "AND use 'web_search' to gather external live data.\n"
            "2. The local documents contain historical Apple inc. financial records. When a user asks for"
            "'sales in my vector store', they are referring to these historical Apple records\n"
            "Do not print raw textual tool calls like '<function\\...>' or explain what you are about to do."
            "Execute the native tool calls directly."
            "When invoking tools, use the native"
            "function-calling interface provided. Do not write out tool calls manually"
            "as plain text strings or use pseudo-HTML/XML tags like <function>."
        )),
        HumanMessage(content=query)]
    print(f"\n\nUser: {query}\n")

    response = llmWithTools.invoke(messages)
    messages.append(response)

    attempts = 0
    while response.tool_calls and attempts < max_retries:
        attempts += 1
        print(f"\n --- Loop iteration {attempts} ---")

        for tool_call in response.tool_calls:
            print(f"🤝 Agent chose: {tool_call['name']}")
            result = validate_and_execute(tool_call= tool_call)
            messages.append(ToolMessage(content=str(result), tool_call_id = tool_call['id']))

        response = llmWithTools.invoke(messages)
        messages.append(response)
    
    print("\n------Final Agent Response ------")
    print(response.content)
    return response.content




In [7]:

# TESTING WITH QUERIES

# Test 2: web-only question
run_research_agent("What is today's date and are the any major apple company news this week?")

# Test 3: requires both.
run_research_agent("Compare the sales in my vector store by year to the apple sales published last year.")

User: What is today's date and are the any major apple company news this week?


 --- Loop iteration 1 ---
🤝 Agent chose: web_search
😁VALIDATOR PASSED: Executing 'web_search' ...

------Final Agent Response ------
Today's date is June 28, 2026, and there have been some major Apple company news this week. Apple has unveiled its new artificial intelligence strategy, including a partnership with Google to use their Gemini models. The company has also made significant improvements and optimizations to its design language, allowing users to adjust transparency and improved text labels and toolbars. Additionally, Apple has announced a new feature called "Spatial Reframing," which uses 3D modeling and AI features to allow users to adjust the angle or composition of an existing photo.
User: Compare the sales in my vector store by year to the apple sales published last year.


 --- Loop iteration 1 ---
🤝 Agent chose: search_local_docs
😁VALIDATOR PASSED: Executing 'search_local_docs' ...
🤝 Agent

'The sales in your vector store by year are as follows:\n\n* 2024: $391,035 million\n* 2023: $383,285 million\n* 2022: $394,328 million\n\nThe Apple sales published last year were:\n\n* 2023: $383 billion\n\nNote that the sales figures are in millions of dollars.'